In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
schemas_p = "/Volumes/workspace/default/retail_volume/retail_db/schemas.json"
schemas_text = spark.read.text(schemas_p ,wholetext = True).first().value
schemas = json.loads(schemas_text)

In [0]:
print(schemas)

{'departments': [{'column_name': 'department_id', 'data_type': 'integer', 'column_position': 1}, {'column_name': 'department_name', 'data_type': 'string', 'column_position': 2}], 'categories': [{'column_name': 'category_id', 'data_type': 'integer', 'column_position': 1}, {'column_name': 'category_department_id', 'data_type': 'integer', 'column_position': 2}, {'column_name': 'category_name', 'data_type': 'string', 'column_position': 3}], 'orders': [{'column_name': 'order_id', 'data_type': 'integer', 'column_position': 1}, {'column_name': 'order_date', 'data_type': 'string', 'column_position': 2}, {'column_name': 'order_customer_id', 'data_type': 'timestamp', 'column_position': 3}, {'column_name': 'order_status', 'data_type': 'string', 'column_position': 4}], 'products': [{'column_name': 'product_id', 'data_type': 'integer', 'column_position': 1}, {'column_name': 'product_cateogry_id', 'data_type': 'integer', 'column_position': 2}, {'column_name': 'product_name', 'data_type': '', 'column

In [0]:
base_dir = "/Volumes/workspace/default/retail_volume/retail_db"

In [0]:
schemas_path = "/Volumes/workspace/default/retail_volume/retail_db/schmeas.json"

In [0]:
def get_column_names(schemas_path, ds_name):
    schemas_text = spark.read.text(schemas_path ,wholetext = True).first().value
    schemas = json.loads(schemas_text)
    column_details = schemas[ds_name]
    col_sorted = [col['column_name'] for col in sorted(column_details, key=lambda col: col['column_position'])]
    return col_sorted

In [0]:
ds_list = ['customers','orders','products','departments','order_items','categories']

In [0]:
tgt_bronze_dir = "/Volumes/workspace/bronze"

In [0]:
import json

for ds in ds_list:
    print(f'processing {ds} data')
    columns = get_column_names(f'{base_dir}/schemas.json',ds)
    df = spark.read.csv(f'{base_dir}/{ds}', inferSchema=False).toDF(*columns)
    df.write.format('delta').mode('overwrite').saveAsTable(f"workspace.bronze.{ds}")

processing customers data
processing orders data
processing products data
processing departments data
processing order_items data
processing categories data


In [0]:
%sql

SHOW TABLES IN workspace.bronze;

database,tableName,isTemporary
bronze,categories,false
bronze,customers,false
bronze,departments,false
bronze,order_items,false
bronze,orders,false
bronze,products,false


In [0]:
orders_df = spark.table("workspace.bronze.products")

display(orders_df.limit(10))

product_id,product_cateogry_id,product_name,product_description,product_price,product_image
1,2,Quest Q64 10 FT. x 10 FT. Slant Leg Instant U,null,59.98,http://images.acmesports.sports/Quest+Q64+10+FT.+x+10+FT.+Slant+Leg+Instant+Up+Canopy
2,2,Under Armour Men's Highlight MC Football Clea,null,129.99,http://images.acmesports.sports/Under+Armour+Men%27s+Highlight+MC+Football+Cleat
3,2,Under Armour Men's Renegade D Mid Football Cl,null,89.99,http://images.acmesports.sports/Under+Armour+Men%27s+Renegade+D+Mid+Football+Cleat
4,2,Under Armour Men's Renegade D Mid Football Cl,null,89.99,http://images.acmesports.sports/Under+Armour+Men%27s+Renegade+D+Mid+Football+Cleat
5,2,Riddell Youth Revolution Speed Custom Footbal,null,199.99,http://images.acmesports.sports/Riddell+Youth+Revolution+Speed+Custom+Football+Helmet
6,2,Jordan Men's VI Retro TD Football Cleat,null,134.99,http://images.acmesports.sports/Jordan+Men%27s+VI+Retro+TD+Football+Cleat
7,2,Schutt Youth Recruit Hybrid Custom Football H,null,99.99,http://images.acmesports.sports/Schutt+Youth+Recruit+Hybrid+Custom+Football+Helmet+2014
8,2,Nike Men's Vapor Carbon Elite TD Football Cle,null,129.99,http://images.acmesports.sports/Nike+Men%27s+Vapor+Carbon+Elite+TD+Football+Cleat
9,2,Nike Adult Vapor Jet 3.0 Receiver Gloves,null,50.0,http://images.acmesports.sports/Nike+Adult+Vapor+Jet+3.0+Receiver+Gloves
10,2,Under Armour Men's Highlight MC Football Clea,null,129.99,http://images.acmesports.sports/Under+Armour+Men%27s+Highlight+MC+Football+Cleat


In [0]:
from pyspark.sql.functions import col,to_timestamp

In [0]:

import json

for ds in ds_list:
    bronze_df = spark.table(f'workspace.bronze.{ds}')
    if ds == 'orders':
        silver_df = (bronze_df
                   .withColumn('order_id',col('order_id').cast('int'))
                   .withColumn('order_customer_id',col('order_customer_id').cast('int'))
                   .withColumn('order_date',to_timestamp("order_date", "yyyy-MM-dd HH:mm:ss.S"))
        )
    elif ds == 'customers':
        silver_df = (bronze_df
                   .withColumn('customer_id',col('customer_id').cast('int'))
        )
    elif ds == 'products':
        silver_df = (bronze_df
                   .withColumn('product_id',col('product_id').cast('int'))
                   .withColumn('product_cateogry_id',col('product_cateogry_id').cast('int'))
                   .withColumn('product_price',col('product_price').cast('double'))
        )
    elif ds == 'departments':
        silver_df = (bronze_df
                   .withColumn('department_id',col('department_id').cast('int'))
        )
    elif ds == 'order_items':
        silver_df = (bronze_df
                   .withColumn('order_item_id',col('order_item_id').cast('int'))
                   .withColumn('order_item_order_id',col('order_item_order_id').cast('int'))
                   .withColumn('order_item_product_id',col('order_item_product_id').cast('int'))
                   .withColumn('order_item_quantity',col('order_item_quantity').cast('int'))
                   .withColumn('order_item_subtotal',col('order_item_subtotal').cast('double'))
                   .withColumn('order_item_product_price',col('order_item_product_price').cast('double'))
        )
    elif ds == 'categories':
        silver_df = (bronze_df
                   .withColumn('category_id',col('category_id').cast('int'))
                   .withColumn('category_department_id',col('category_department_id').cast('int'))
        )

    print(f'processing {ds} data')
    silver_df.write.format('delta').mode('overwrite').saveAsTable(f"workspace.silver.{ds}")


processing customers data
processing orders data
processing products data
processing departments data
processing order_items data
processing categories data


In [0]:
%sql

SHOW TABLES IN workspace.silver;

database,tableName,isTemporary
silver,categories,false
silver,customers,false
silver,departments,false
silver,order_items,false
silver,orders,false
silver,products,false


In [0]:
%sql
create or replace table workspace.gold.daily_sales
using delta
as
 select cast(o.order_date as date) as order_date,
 sum(oi.order_item_subtotal) as total_sales,
 count(distinct o.order_id) as total_orders,
 SUM(oi.order_item_subtotal) / COUNT(DISTINCT o.order_id)
        AS average_order_value


 from workspace.bronze.orders as o
 join workspace.bronze.order_items as oi
 on oi.order_item_order_id = o.order_id
 group by 1


num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.gold.daily_sales


order_date,total_sales,total_orders,average_order_value
2013-09-20,82662.51000000004,119,694.6429411764709
2013-09-25,141775.64000000007,242,585.849752066116
2013-10-14,61613.75000000004,115,535.7717391304351
2013-10-27,106982.80999999998,184,581.4283152173912
2013-11-10,83681.05999999995,146,573.1579452054791
2013-12-27,87067.14999999995,138,630.9213768115939
2014-01-06,82307.97999999998,130,633.1383076923075
2014-02-07,125604.3399999999,199,631.177587939698
2014-02-22,64485.77000000003,108,597.0904629629632
2014-02-26,69276.28000000004,123,563.2217886178865


In [0]:
%sql

create or replace table workspace.gold.sales_by_category
using delta
as
select 
c.category_id,
c.category_name,
SUM(oi.order_item_subtotal) as total_sales,
SUM(oi.order_item_quantity) as quantity_sold,
COUNT(DISTINCT oi.order_item_order_id) as total_orders

from workspace.silver.categories as c
join workspace.silver.products as p
on p.product_cateogry_id = c.category_id
join workspace.silver.order_items as oi
on p.product_id = oi.order_item_product_id
group by 1,2

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.gold.sales_by_category

category_id,category_name,total_sales,quantity_sold,total_orders
16,As Seen on TV!,20597.939999999995,206,68
43,Camping & Hiking,4118425.4199997927,13729,12299
17,Cleats,4431942.659999762,73734,20386
6,Tennis & Racquet,44585.09,991,327
2,Soccer,26477.04999999998,295,136
34,Golf Bags & Carts,10369.389999999992,61,61
11,Fitness Accessories,35601.43999999997,856,307
26,Girls' Apparel,151706.20000000007,3615,1191
45,Fishing,6929653.499999715,17325,15164
35,Golf Gloves,116695.38999999985,2961,1064


In [0]:
%sql

create or replace table workspace.gold.top_products
using delta
as
select 
p.product_id,
p.product_name,
SUM(oi.order_item_subtotal) as total_sales,
SUM(oi.order_item_quantity) as quantity_sold,
COUNT(DISTINCT oi.order_item_order_id) as total_orders

from workspace.silver.products as p
join workspace.silver.order_items as oi
on p.product_id = oi.order_item_product_id
group by 1,2

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.gold.top_products

product_id,product_name,total_sales,quantity_sold,total_orders
134,Nike Women's Legend V-Neck T-Shirt,20025.0,801,278
172,Nike Women's Tempo Shorts,27210.0,907,318
885,Team Golf St. Louis Cardinals Putter Grip,23940.419999999955,958,313
926,Glove It Imperial Golf Towel,14870.699999999979,930,302
858,GolfBuddy VT3 GPS Watch,11799.409999999993,59,59
19,Nike Men's Fingertrap Max Training Shoe,7999.359999999993,64,63
58,Diamondback Boys' Insight 24 Performance Hybr,8699.71,29,28
860,Bushnell Pro X7 Jolt Slope Rangefinder,6599.889999999999,11,8
564,Nike Men's Deutschland Weltmeister Winners Bl,27210.0,907,297
703,Top Flite Women's 2014 XL Hybrid,17731.12999999997,887,303


In [0]:
%sql

SHOW TABLES IN workspace.gold;

database,tableName,isTemporary
gold,daily_sales,false
gold,sales_by_category,false
gold,top_products,false


In [0]:
%sql

select nvl2()

In [0]:
%sql
SELECT 'daily_sales' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.daily_sales

UNION ALL

SELECT 'sales_by_category', COUNT(*)
FROM workspace.gold.sales_by_category

UNION ALL

SELECT 'top_products', COUNT(*)
FROM workspace.gold.top_products;

table_name,row_count
daily_sales,364
sales_by_category,33
top_products,100


In [0]:
%sql
SELECT
    COALESCE(total_sales, 0) AS total_sales
FROM workspace.gold.daily_sales;

total_sales
82662.51000000004
141775.64000000007
61613.75000000004
106982.80999999998
83681.05999999995
87067.14999999995
82307.97999999998
125604.3399999999
64485.77000000003
69276.28000000004


In [0]:
%sql
SELECT
    COALESCE(total_sales, 0) AS total_sales
FROM workspace.gold.sales_by_category;

total_sales
20597.939999999995
4118425.4199997927
4431942.659999762
44585.09
26477.04999999998
10369.389999999992
35601.43999999997
151706.20000000007
6929653.499999715
116695.38999999985


In [0]:
%sql
SELECT
    COALESCE(total_sales, 0) AS total_sales
FROM workspace.gold.top_products;

total_sales
20025.0
27210.0
23940.419999999955
14870.699999999979
11799.409999999993
7999.359999999993
8699.71
6599.889999999999
27210.0
17731.12999999997


In [0]:
%sql
SELECT SUM(total_sales) AS daily_sales_total
FROM workspace.gold.daily_sales;

SELECT SUM(total_sales) AS category_sales_total
FROM workspace.gold.sales_by_category;

SELECT SUM(total_sales) AS product_sales_total
FROM workspace.gold.top_products;

daily_sales_total
3.432261992999999E7
